# SPAI Glove Detection — YOLOv8 Training Notebook

**Sprint:** Week 2  
**Goal:** Train a YOLOv8n model to detect gloves on first-person camera frames as the v1 baseline for SPAI's computer vision pipeline.

## What this notebook does
1. Sets up Colab with the Ultralytics library and Kaggle API
2. Downloads the PPE dataset (`shlokraval/ppe-dataset-yolov8`)
3. Filters labels to glove-only and remaps class id to 0
4. Trains YOLOv8n on the filtered dataset
5. Validates the model and exports to ONNX

## Requirements
- Colab runtime: T4 GPU (Runtime → Change runtime type)
- A `kaggle.json` API token (from kaggle.com → Settings → API → Create Legacy API Key)

## Caveats
This trains on **construction PPE data**, not medical. v1 validates the pipeline; v2+ will fine-tune on medical-domain data. See `docs/dataset-notes.md` for the full reasoning.

## Step 1 — Verify GPU

In [ ]:
!nvidia-smi

## Step 2 — Install Ultralytics

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## Step 3 — Upload Kaggle credentials

When the cell runs, click 'Choose Files' and select your `kaggle.json` file.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## Step 4 — Download and unzip the dataset

In [ ]:
!kaggle datasets download -d shlokraval/ppe-dataset-yolov8
!mkdir -p /content/datasets/ppe_raw
!unzip -q ppe-dataset-yolov8.zip -d /content/datasets/ppe_raw
!ls /content/datasets/ppe_raw

## Step 5 — Inspect the source class list

Confirm which class id corresponds to `Gloves` before filtering.

In [ ]:
!cat /content/datasets/ppe_raw/data.yaml

## Step 6 — Filter to glove-only and build YOLOv8 dataset structure

The source has 14 classes. We keep only `Gloves` (class id 1 in the source) and remap it to class id 0 in the new dataset. Roboflow's `valid` folder is renamed to `val` to match YOLOv8's default convention.

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/datasets/ppe_raw")
DST = Path("/content/datasets/glove_detection")
GLOVE_SRC_ID = 1  # 'Gloves' in the source data.yaml

splits = {"train": "train", "valid": "val", "test": "test"}

for src_split, dst_split in splits.items():
    (DST / "images" / dst_split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / dst_split).mkdir(parents=True, exist_ok=True)

    for img in (SRC / src_split / "images").glob("*"):
        shutil.copy(img, DST / "images" / dst_split / img.name)

    for lbl in (SRC / src_split / "labels").glob("*.txt"):
        kept = []
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if parts and int(parts[0]) == GLOVE_SRC_ID:
                kept.append("0 " + " ".join(parts[1:]))
        (DST / "labels" / dst_split / lbl.name).write_text("\n".join(kept))

print("Done filtering.")
print("Train images:", len(list((DST / "images" / "train").glob("*"))))
print("Val images:  ", len(list((DST / "images" / "val").glob("*"))))
print("Test images: ", len(list((DST / "images" / "test").glob("*"))))

## Step 7 — Write the data.yaml config

In [ ]:
yaml_content = """path: /content/datasets/glove_detection
train: images/train
val: images/val
test: images/test

names:
  0: glove
"""

with open("/content/datasets/glove_detection/data.yaml", "w") as f:
    f.write(yaml_content)

!cat /content/datasets/glove_detection/data.yaml

## Step 8 — Smoke test (3 epochs)

Validates the pipeline before committing to a long training run. Should take ~10–15 minutes on a T4.

In [ ]:
model = YOLO("yolov8n.pt")

smoke_results = model.train(
    data="/content/datasets/glove_detection/data.yaml",
    epochs=3,
    imgsz=640,
    batch=16,
    name="spai_glove_smoketest"
)

## Step 9 — Full training run (15 epochs, with early stopping)

If smoke test mAP50 looked reasonable (above ~0.5), proceed with the full run. ~75–90 min on T4.

`patience=5` means training stops early if validation mAP doesn't improve for 5 epochs.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/datasets/glove_detection/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    name="spai_glove_detection_v1",
    patience=5
)

## Step 10 — Validate and inspect outputs

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
from IPython.display import Image, display
import os

run_dir = "/content/runs/detect/spai_glove_detection_v1"
for f in sorted(os.listdir(run_dir)):
    print(" ", f)

In [ ]:
# Display the headline plots
for plot in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    path = f"{run_dir}/{plot}"
    if os.path.exists(path):
        display(Image(path))

## Step 11 — Run inference on test images

In [ ]:
model.predict(
    source="/content/datasets/glove_detection/images/test",
    save=True,
    conf=0.25
)

## Step 12 — Export to ONNX

ONNX format will be used by the backend model service. CoreML conversion (for on-device AVP inference, future) is also possible from here.

In [ ]:
model.export(format="onnx")